In [ ]:
import pandas as pd
import os
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("Avvio confronto Mean vs Median per Random Forest Ottimizzato (BAR 1,2)...")

# Dati e Modello

data_dir_mean = '/content/data/fingerprints/'
data_dir_median = '/content/drive/My Drive/Tesi Magistrale/Data_Median/'

train_file_mean = os.path.join(data_dir_mean, 'android_session_1_1.csv')
test_file_mean = os.path.join(data_dir_mean, 'android_session_2_2.csv')

train_file_median = os.path.join(data_dir_median, 'android_session_1_1_median.csv')
test_file_median = os.path.join(data_dir_median, 'android_session_2_2_median.csv')

# Parametri ottimizzati del Random Forest
rf_champion_params = {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}

# Funzione helper per eseguire un esperimento
def run_rf_experiment(train_path, test_path, model_params):
    # Carica dati
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Prepara dati
    X_train = train_df.drop(columns=['room', 'artwork'])
    y_train_room = train_df['room']
    y_train_artwork = train_df['artwork']
    X_test = test_df.drop(columns=['room', 'artwork'])
    y_test_room = test_df['room']
    y_test_artwork = test_df['artwork']

    artwork_encoder = LabelEncoder()
    y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
    y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)

    # Riempi eventuali NaN rimasti nei dati (importante!)
    X_train.fillna(110.0, inplace=True)
    X_test.fillna(110.0, inplace=True)

    results = {}

    # Addestra e valuta per STANZA
    rf_room = RandomForestClassifier(random_state=42, n_jobs=-1, **model_params)
    rf_room.fit(X_train, y_train_room)
    results['Room Accuracy'] = accuracy_score(y_test_room, rf_room.predict(X_test)) * 100

    # Addestra e valuta per OPERA D'ARTE
    rf_artwork = RandomForestClassifier(random_state=42, n_jobs=-1, **model_params)
    rf_artwork.fit(X_train, y_train_artwork_encoded)
    results['Artwork Top-1 Accuracy'] = accuracy_score(y_test_artwork_encoded, rf_artwork.predict(X_test)) * 100

    artwork_probabilities = rf_artwork.predict_proba(X_test)
    top3_predictions = artwork_probabilities.argsort(axis=1)[:, -3:]
    correct_top3 = [y_test_artwork_encoded[j] in top3_predictions[j] for j in range(len(y_test_artwork_encoded))]
    results['Artwork Top-3 Accuracy'] = (sum(correct_top3) / len(correct_top3)) * 100

    return results

# Esegui gli esperimenti

print("\nEsecuzione su dati MEAN...")
start_mean = time.time()
results_mean = run_rf_experiment(train_file_mean, test_file_mean, rf_champion_params)
end_mean = time.time()
print(f"Completato in {end_mean - start_mean:.2f} secondi.")

print("\nEsecuzione su dati MEDIAN...")
start_median = time.time()
results_median = run_rf_experiment(train_file_median, test_file_median, rf_champion_params)
end_median = time.time()
print(f"Completato in {end_median - start_median:.2f} secondi.")

# Confronta e Salva

comparison_data = {
    'Fingerprint Type': ['Mean (Original)', 'Median (New)'],
    'Room Accuracy': [results_mean['Room Accuracy'], results_median['Room Accuracy']],
    'Artwork Top-1 Accuracy': [results_mean['Artwork Top-1 Accuracy'], results_median['Artwork Top-1 Accuracy']],
    'Artwork Top-3 Accuracy': [results_mean['Artwork Top-3 Accuracy'], results_median['Artwork Top-3 Accuracy']]
}
comparison_df = pd.DataFrame(comparison_data)

output_folder = "/content/drive/My Drive/Tesi Magistrale/Confronto_Mean_Median/"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "comparison_BAR_1_2.csv")
comparison_df.to_csv(output_file, index=False, float_format='%.2f')

print("\n\n--- CONFRONTO MEAN vs MEDIAN (BAR 1,2 - RF Ottimizzato) ---")
print(comparison_df.to_string(index=False))
print(f"\n Confronto salvato in: {output_file}")

In [ ]:
import pandas as pd
import os
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("Avvio confronto Sensor Fusion vs. Solo RSSI (Median) per RF Ottimizzato (BAR 1,2)...")

# Dati e Modello

# Percorsi dei file
data_dir_median = '/content/drive/My Drive/Tesi Magistrale/Data_Median/' # Dati Mediana (solo RSSI)
data_dir_fusion = '/content/drive/My Drive/Tesi Magistrale/Data_SensorFusion/' # Dati Sensor Fusion

train_file_median = os.path.join(data_dir_median, 'android_session_1_1_median.csv')
test_file_median = os.path.join(data_dir_median, 'android_session_2_2_median.csv')

train_file_fusion = os.path.join(data_dir_fusion, 'android_session_1_1_fusion_median.csv')
test_file_fusion = os.path.join(data_dir_fusion, 'android_session_2_2_fusion_median.csv')


# Parametri ottimizzati del Random Forest
rf_champion_params = {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}

# Funzione helper per eseguire un esperimento
def run_rf_experiment(train_path, test_path, model_params):
    # Carica dati
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Prepara dati
    X_train = train_df.drop(columns=['room', 'artwork'])
    y_train_room = train_df['room']
    y_train_artwork = train_df['artwork']
    X_test = test_df.drop(columns=['room', 'artwork'])
    y_test_room = test_df['room']
    y_test_artwork = test_df['artwork']

    artwork_encoder = LabelEncoder()
    y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
    try:
        y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)
    except ValueError as e:
         print(f"Attenzione: Etichette nel test set non presenti nel training set: {e}")
         known_labels_mask = y_test_artwork.isin(artwork_encoder.classes_)
         X_test = X_test[known_labels_mask]
         y_test_room = y_test_room[known_labels_mask]
         y_test_artwork = y_test_artwork[known_labels_mask]
         y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)


    # Gestione Robusta dei NaN
    # Identifica colonne RSSI e sensori
    rssi_cols = [col for col in X_train.columns if '-' in col]
    sensor_cols = [col for col in X_train.columns if '-' not in col]

    X_train[rssi_cols] = X_train[rssi_cols].fillna(110.0)
    X_test[rssi_cols] = X_test[rssi_cols].fillna(110.0)
    X_train[sensor_cols] = X_train[sensor_cols].fillna(0.0)
    X_test[sensor_cols] = X_test[sensor_cols].fillna(0.0)

    results = {}

    # Addestra e valuta per STANZA
    rf_room = RandomForestClassifier(random_state=42, n_jobs=-1, **model_params)
    rf_room.fit(X_train, y_train_room)
    results['Room Accuracy'] = accuracy_score(y_test_room, rf_room.predict(X_test)) * 100

    # Addestra e valuta per OPERA D'ARTE
    rf_artwork = RandomForestClassifier(random_state=42, n_jobs=-1, **model_params)
    rf_artwork.fit(X_train, y_train_artwork_encoded)
    results['Artwork Top-1 Accuracy'] = accuracy_score(y_test_artwork_encoded, rf_artwork.predict(X_test)) * 100

    artwork_probabilities = rf_artwork.predict_proba(X_test)
    top3_predictions = artwork_probabilities.argsort(axis=1)[:, -3:]
    correct_top3 = [y_test_artwork_encoded[j] in top3_predictions[j] for j in range(len(y_test_artwork_encoded))]
    results['Artwork Top-3 Accuracy'] = (sum(correct_top3) / len(correct_top3)) * 100

    return results

# Esegui gli esperimenti

print("\nEsecuzione su dati MEDIAN (Solo RSSI)...")
start_median = time.time()
results_median_only = run_rf_experiment(train_file_median, test_file_median, rf_champion_params)
end_median = time.time()
print(f"Completato in {end_median - start_median:.2f} secondi.")

print("\nEsecuzione su dati SENSOR FUSION (RSSI Mediana + Sensori)...")
start_fusion = time.time()
results_fusion = run_rf_experiment(train_file_fusion, test_file_fusion, rf_champion_params)
end_fusion = time.time()
print(f"Completato in {end_fusion - start_fusion:.2f} secondi.")


# Confronta e Salva

comparison_data = {
    'Feature Set': ['RSSI Median Only', 'Sensor Fusion (RSSI Median + Sensors)'],
    'Room Accuracy': [results_median_only['Room Accuracy'], results_fusion['Room Accuracy']],
    'Artwork Top-1 Accuracy': [results_median_only['Artwork Top-1 Accuracy'], results_fusion['Artwork Top-1 Accuracy']],
    'Artwork Top-3 Accuracy': [results_median_only['Artwork Top-3 Accuracy'], results_fusion['Artwork Top-3 Accuracy']]
}
comparison_df = pd.DataFrame(comparison_data)

output_folder = "/content/drive/My Drive/Tesi Magistrale/Confronto_SensorFusion/"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "comparison_fusion_vs_median_BAR_1_2.csv")
comparison_df.to_csv(output_file, index=False, float_format='%.2f')

print("\n\n--- CONFRONTO SENSOR FUSION vs SOLO RSSI (BAR 1,2 - RF Ottimizzato) ---")
print(comparison_df.to_string(index=False))
print(f"\n Confronto salvato in: {output_file}")